# Dimensions Materialize — Trigger & Verify

Triggers the `dimensions_materialize` OGC Process and verifies the `_dimensions_` catalog over HTTP only.

**Why HTTP-only?** The notebook does not import `ogc_dimensions`. That package is only installed when the server image is built with `SCOPE` including `extension_dimensions` (see `pyproject.toml` — distribution `ogc-dimensions` from `git+https://github.com/ccancellieri/ogc-dimensions.git@main#subdirectory=reference-implementation`, import name `ogc_dimensions`). As long as the *server* has it, the notebook works from any kernel.

**If you are developing ogc-dimensions from local source:** install it into the running server container (`uv pip install -e /path/to/ogc-dimensions/reference-implementation`) and restart the service — do not install into the notebook kernel.

In [ ]:
import os, time, json
import httpx

BASE_URL = os.environ.get('DYNASTORE_BASE_URL', 'http://localhost:8000')
TOKEN = os.environ.get('DYNASTORE_TOKEN')  # optional bearer token
HEADERS = {'Accept': 'application/json'}
if TOKEN:
    HEADERS['Authorization'] = f'Bearer {TOKEN}'

PROCESS_ID = 'dimensions_materialize'
DIMENSIONS_CATALOG = '_dimensions_'
print('Base URL:', BASE_URL)

## 1. Preflight
Confirms the task is registered server-side and the dimensions extension is wired in. If either check fails, rebuild the server image with `SCOPE` including `extension_dimensions` and `task_dimensions_materialize`.

In [ ]:
r = httpx.get(f'{BASE_URL}/processes/{PROCESS_ID}', headers=HEADERS)
assert r.status_code == 200, (
    f'Process {PROCESS_ID} not registered (HTTP {r.status_code}). '
    'Rebuild the server with SCOPE including task_dimensions_materialize.'
)
print(json.dumps(r.json(), indent=2)[:600])

In [ ]:
r = httpx.get(f'{BASE_URL}/collections', params={'catalog': DIMENSIONS_CATALOG}, headers=HEADERS)
assert r.status_code == 200, (
    f'Catalog {DIMENSIONS_CATALOG} lookup failed (HTTP {r.status_code}). '
    'The dimensions extension may not be loaded — rebuild with SCOPE including extension_dimensions.'
)
collections = r.json().get('collections', [])
print(f'Found {len(collections)} dimension collections already materialized:')
for c in collections:
    extra = (c.get('extra_metadata') or {})
    cube = extra.get('cube:dimensions') or {}
    print(f"  - {c.get('id')}: extent_keys={list(cube.keys())[:5]}")

## 2. Trigger the task
Pass `dim_names` to materialize a subset, or leave empty to process all. Set `force=True` to bypass the `cube:dimensions` equality skip.

In [ ]:
payload = {
    'inputs': {
        # 'dim_names': ['time', 'altitude'],
        'force': False,
    }
}
r = httpx.post(
    f'{BASE_URL}/processes/{PROCESS_ID}/execution',
    headers={**HEADERS, 'Prefer': 'respond-async', 'Content-Type': 'application/json'},
    json=payload,
)
r.raise_for_status()
job = r.json()
job_id = job.get('jobID') or job.get('id') or r.headers.get('Location', '').rsplit('/', 1)[-1]
print('Job ID:', job_id)
print(json.dumps(job, indent=2)[:400])

## 3. Poll until complete

In [ ]:
TERMINAL = {'successful', 'failed', 'dismissed'}
while True:
    r = httpx.get(f'{BASE_URL}/jobs/{job_id}', headers=HEADERS)
    r.raise_for_status()
    s = r.json()
    status = s.get('status')
    progress = s.get('progress')
    msg = s.get('message', '')
    print(f'status={status} progress={progress} {msg}')
    if status in TERMINAL:
        break
    time.sleep(2)

## 4. Fetch results

In [ ]:
r = httpx.get(f'{BASE_URL}/jobs/{job_id}/results', headers=HEADERS)
r.raise_for_status()
results = r.json()
print(json.dumps(results, indent=2)[:2000])

## 5. Verify: list dimensions catalog after run

In [ ]:
r = httpx.get(f'{BASE_URL}/collections', params={'catalog': DIMENSIONS_CATALOG}, headers=HEADERS)
r.raise_for_status()
for c in r.json().get('collections', []):
    cid = c.get('id')
    it = httpx.get(f'{BASE_URL}/collections/{cid}/items', params={'limit': 1}, headers=HEADERS)
    total = it.json().get('numberMatched') if it.is_success else '?'
    print(f'  {cid}: numberMatched={total}')

## 6. Optional: force re-run
Bypass the `cube:dimensions` skip — re-upserts every member even if unchanged.

In [ ]:
r = httpx.post(
    f'{BASE_URL}/processes/{PROCESS_ID}/execution',
    headers={**HEADERS, 'Prefer': 'respond-async', 'Content-Type': 'application/json'},
    json={'inputs': {'force': True}},
)
r.raise_for_status()
print('Force re-run triggered:', r.json())